In [41]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder,StandardScaler

In [42]:
df = pd.read_csv('../0.dataset/dataset_penjualan_emas_kotor.csv')
df.head(5)

,Transaction_ID,Date,Customer_ID,Gold_Type,Karat,Weight_Gram,Price_Per_Gram,Total_Price,Payment_Method,Store_Location
0,TXN-0295,2024-04-21,CUST-480,Perhiasan,24,35.55,1370210,48710965,Kartu Kredit,JKT Pusat
1,TXN-0809,2024-03-09,CUST-373,Perhiasan,18,NaN,961081,11215815,Transfer Bank,Makassar
2,TXN-0364,2024-11-18,CUST-175,Perhiasan,24,49.59,1323332,65624033,QRIS,Bandung
3,TXN-0869,2025-09-30,CUST-313,Perhiasan,24,44.73,1352769,60509357,QRIS,Makassar
4,TXN-0326,2025-03-13,CUST-158,Koin,18,10.93,960581,10499150,Transfer Bank,Surabaya


In [43]:
df = df.dropna()
df.isnull().sum()

Transaction_ID    0
Date              0
Customer_ID       0
Gold_Type         0
Karat             0
Weight_Gram       0
Price_Per_Gram    0
Total_Price       0
Payment_Method    0
Store_Location    0
dtype: int64

In [44]:
df = df.drop_duplicates(keep='last')
df.duplicated().sum()

np.int64(0)

In [45]:
df = df.drop(columns=['Transaction_ID','Customer_ID','Date'])
df.head()

,Gold_Type,Karat,Weight_Gram,Price_Per_Gram,Total_Price,Payment_Method,Store_Location
0,Perhiasan,24,35.55,1370210,48710965,Kartu Kredit,JKT Pusat
2,Perhiasan,24,49.59,1323332,65624033,QRIS,Bandung
3,Perhiasan,24,44.73,1352769,60509357,QRIS,Makassar
5,Koin,24,38.79,1350607,52390045,Kartu Kredit,Makassar
6,Batangan,24,34.33,1362814,46785404,QRIS,Medan


In [46]:
colom_numeric = df.select_dtypes(include=np.number).columns
colom_categorical = df.select_dtypes(include='object').columns

In [47]:
df[colom_categorical] = df[colom_categorical].apply(lambda x:x.str.title())

In [48]:
label = LabelEncoder()
for col in colom_categorical:
    df[col] = label.fit_transform(df[col])

## 1.Feature Selection (Pemilihan Fitur)

### 1.Menghapus Fitur dengan Varians Rendah

In [49]:
from sklearn.feature_selection import VarianceThreshold
df_target = df.copy()

colom_target = 'Gold_Type'
df_x = df_target.drop(columns=colom_target)
df_y = df_target[colom_target]

selector = VarianceThreshold(threshold=0.1)
selector.fit(df_x)

selected_features = df_x.columns[selector.get_support()]

df_x = df_x[selected_features]
kolom_numeric_tersisa = [col for col in selected_features if col in colom_numeric]

scaler = StandardScaler()
df_target = df_x.copy()
df_target[kolom_numeric_tersisa] = scaler.fit_transform(df_target[kolom_numeric_tersisa])
df_target[colom_target] = df_y

df_target.head()

,Karat,Weight_Gram,Price_Per_Gram,Total_Price,Payment_Method,Store_Location,Gold_Type
0,1.117092,0.737917,1.387783,0.202386,0,2,2
2,1.117092,1.696567,1.105956,0.432261,1,0,2
3,1.117092,1.364727,1.282929,0.362745,1,3,2
5,1.117092,0.959144,1.269932,0.252390,0,3,1
6,1.117092,0.654615,1.343319,0.176215,1,4,0


### 2.Menghapus Fitur yang Sangat Berkorelasi

In [50]:
df_target = df.copy()

colom_target = 'Gold_Type'
df_x = df_target.drop(columns=colom_target)
df_y = df_target[colom_target]

cor_matrix = df_x.corr().abs()
upper = cor_matrix.where(np.triu(np.ones(cor_matrix.shape),k=1).astype(bool))

threshold = 0.9
to_drop = [column for column in upper.columns if any(upper[column] > threshold)]

df_target = df_x.drop(columns=to_drop)
df_target[colom_target] = df_y

df_target.head()

,Karat,Weight_Gram,Total_Price,Payment_Method,Store_Location,Gold_Type
0,24,35.55,48710965,0,2,2
2,24,49.59,65624033,1,0,2
3,24,44.73,60509357,1,3,2
5,24,38.79,52390045,0,3,1
6,24,34.33,46785404,1,4,0


## 2.Menggunakan Metode Wrapper untuk Seleksi Fitur

### 1.Forward Selection

In [51]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
df_target = df.copy()

colom_target = 'Gold_Type'
df_x = df_target.drop(columns=colom_target)
df_y = df_target[colom_target]

scaler = StandardScaler()
df_x[colom_numeric] = scaler.fit_transform(df_x[colom_numeric])

model_logistic = LogisticRegression(max_iter=1000)
sfs = SequentialFeatureSelector(model_logistic,n_features_to_select=4,direction='forward',cv=4)
sfs = sfs.fit(df_x,df_y)

selected_features = list(sfs.get_feature_names_out())
df_target = df_x[selected_features]

df_target.head()


,Karat,Price_Per_Gram,Total_Price,Store_Location
0,1.117092,1.387783,0.202386,2
2,1.117092,1.105956,0.432261,0
3,1.117092,1.282929,0.362745,3
5,1.117092,1.269932,0.252390,3
6,1.117092,1.343319,0.176215,4


### 2.Backward Elimination

In [52]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

df_target = df.copy()

colom_target = 'Gold_Type'
df_x = df_target.drop(columns=colom_target)
df_y = df_target[colom_target]

scaler = StandardScaler()
df_x[colom_numeric] = scaler.fit_transform(df_x[colom_numeric])

model_logistic = LogisticRegression(max_iter=1000)
sfs = SequentialFeatureSelector(model_logistic,n_features_to_select=4,direction='backward',cv=4)
sfs = sfs.fit(df_x,df_y)

selected_features = list(sfs.get_feature_names_out())
df_target = df_x[selected_features]

df_target.head()

,Weight_Gram,Price_Per_Gram,Payment_Method,Store_Location
0,0.737917,1.387783,0,2
2,1.696567,1.105956,1,0
3,1.364727,1.282929,1,3
5,0.959144,1.269932,0,3
6,0.654615,1.343319,1,4


## 3.Menggunakan Metode Embedded untuk Seleksi Fitur

### 1.Elastic Net Selection

In [53]:
from sklearn.linear_model import ElasticNet
df_target = df.copy()

colom_target = 'Gold_Type'
df_x = df_target.drop(columns=colom_target)
df_y = df_target[colom_target]

elastic_net = ElasticNet(alpha=0.1,l1_ratio=0.5)
elastic_net.fit(df_x,df_y)

selected_features = df_x.columns[elastic_net.coef_ != 0].to_list()
df_target = df_x[selected_features]

df_target.head()

,Weight_Gram,Price_Per_Gram,Total_Price,Store_Location
0,35.55,1370210,48710965,2
2,49.59,1323332,65624033,0
3,44.73,1352769,60509357,3
5,38.79,1350607,52390045,3
6,34.33,1362814,46785404,4


### 2.Random Forest Feature Importance

In [57]:
from sklearn.tree import DecisionTreeClassifier
df_target = df.copy()

colom_target = 'Gold_Type'
df_x = df_target.drop(columns=colom_target)
df_y = df_target[colom_target]

tree = DecisionTreeClassifier(random_state=42)
tree = tree.fit(df_x,df_y)

selected_features = df_x.columns[tree.feature_importances_  > 0.01 ].to_list()
df_target = df_x[selected_features]

df_target.head()


,Weight_Gram,Price_Per_Gram,Total_Price,Payment_Method,Store_Location
0,35.55,1370210,48710965,0,2
2,49.59,1323332,65624033,1,0
3,44.73,1352769,60509357,1,3
5,38.79,1350607,52390045,0,3
6,34.33,1362814,46785404,1,4
